# RunResult, límites y preparación

Resultado serializable y linaje en todos los límites de ejecución.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [ ]:
import agentic_systems as toolkit


In [ ]:
@toolkit.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = toolkit.Action("increase", "increment")
multiply = toolkit.Action("multiply", "double")

In [ ]:
import json
agent = toolkit.Agent("calculator", toolkit.Sequence((increase, multiply)), skills=(arithmetic,))
compiled = agent.compile()
result = compiled.run({"value": 3}, limits=toolkit.Limits(max_actions=2, timeout_s=10))
assert result.ok
encoded = json.loads(json.dumps(result.to_dict(), allow_nan=False))
assert toolkit.RunResult(**encoded).data == {"value": 8}
for node in result.walk():
    for child in node.children:
        assert child.parent_execution_id == node.execution_id
limited = agent.run({"value": 3}, limits=toolkit.Limits(max_actions=1))
assert not limited.ok
print({"output": result.data, "lineage_nodes": len(list(result.walk())), "limit_detected": not limited.ok})

In [ ]:
assert (await agent.arun({'value':3})).data == result.data
assert (await compiled.arun({'value':3})).data == result.data
assert result.check_invariants().ok
assert result.trace('full') == result.to_dict()
memory = result.lineage(name='calculator',question='What happened?',goal='Explain',tags=['offline'],metadata={'tutorial':7},max_tool_rows=2)
print(memory.explain())
print(memory.to_prompt_context(max_chars=500))
print(result.trace())

to_dict es el esquema de intercambio; RunResult(**payload) restaura ese contrato. walk recorre la jerarquía, trace resume ejecución y lineage proyecta evidencia para explicación. Los contadores de Tools e inferencia son distintos. raw_responses no contiene objetos SDK ni credenciales. Los timeouts son cooperativos, no cancelación fuerte.

`compile()` no ejecuta Tools ni condiciones. `run()` lo llama internamente. Ambos caminos devuelven el mismo contrato; compile es preparación opcional, no otro constructor. Los límites son cooperativos y comparten reservas entre componentes; no matan un callback Python bloqueado. Agent/System también ofrecen `arun` con el mismo resultado.